# FloodAI v0.1 — Flood Occurrence Pipeline (Driver Notebook)

**This notebook contains no modeling logic.** It only orchestrates calls into
the `floodai` package under `src/`. If you find yourself wanting to edit a
formula, threshold, or feature definition here — stop, and edit the
corresponding module in `src/floodai/` instead, then re-run this notebook.
That separation is what makes the project testable and reviewable.

## Before running

1. This pipeline reports **held-out metrics only**. There is no code path in
   this package that can produce a "nicer" training-inclusive number labeled
   as a headline result — `evaluation/metrics.py` enforces this at the type
   level, not by convention.
2. The IMD data provider has not been executed against the live IMD server
   from the environment this code was authored in. **Run
   `notebooks/validate_first_run.py` first** and read its output before
   trusting any downstream number in this notebook.
3. Three basins are studied: Ganga (Bihar), Brahmaputra (Assam), Mahanadi
   (Odisha). Mahanadi is included specifically as a hard case — if it
   produces near-zero F1, that is a reportable finding (consistent with the
   reservoir/cyclone-driven flood mechanism documented in `config/config.yaml`),
   not a bug to hide.


In [ ]:
# Setup — run once per Colab session
!pip install -q -r requirements.txt
!pip install -q -e .


import sys
sys.path.insert(0, "src")

import logging
from pathlib import Path

from floodai.config import load_config
from floodai.logging_utils import setup_logging, write_run_manifest

CONFIG_PATH = Path("config/config.yaml")
cfg = load_config(CONFIG_PATH)
logger = setup_logging(cfg.output_dir, run_name=cfg.experiment_id)
manifest_path = write_run_manifest(cfg.output_dir, cfg.experiment_id, CONFIG_PATH, cfg.random_seed)
logger.info("Run manifest written to %s", manifest_path)


!python notebooks/validate_first_run.py
# If this prints [FAIL] anywhere above, STOP and resolve before continuing.


from floodai.gis.points import generate_grid_fallback_points, basin_points_to_dataframe

# NOTE: using the deterministic grid fallback here because no administrative
# boundary shapefile is bundled with this package (geopandas + a GADM/OSM
# India boundaries file is required for the preferred admin-centroid method —
# see floodai.gis.points.generate_admin_centroid_points). If you have access
# to an India district/sub-district boundary file, swap this call for that
# function and re-run — it is a one-line change, by design.
all_points = []
for basin_key, basin_cfg in cfg.basins.items():
    pts = generate_grid_fallback_points(
        basin_key=basin_key,
        bbox=basin_cfg.bbox,
        n_points_target=basin_cfg.n_points_target,
        seed=cfg.random_seed,
    )
    all_points.extend(pts)

points_df = basin_points_to_dataframe(all_points)
points_df.to_csv(cfg.output_dir / "basin_points.csv", index=False)
print(f"Generated {len(points_df)} points across {points_df['basin_key'].nunique()} basins")
points_df.groupby("basin_key").size()


from floodai.data.rainfall_providers import get_rainfall_provider
import pandas as pd
import time

provider = get_rainfall_provider(
    cfg.raw["data_sources"]["rainfall"]["provider"],
)
start_year = cfg.raw["data_sources"]["rainfall"]["start_year"]
end_year = cfg.raw["data_sources"]["rainfall"]["end_year"]

all_series = []
for i, row in points_df.iterrows():
    try:
        df_point = provider.fetch_point_series(
            row["lat"], row["lon"], f"{start_year}0101", f"{end_year}1231"
        )
        df_point["point_id"] = row["point_id"]
        df_point["basin_key"] = row["basin_key"]
        all_series.append(df_point)
    except Exception as e:
        logger.error("Failed to fetch point %s: %s", row["point_id"], e)
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(points_df)} points fetched...")

rainfall_df = pd.concat(all_series, ignore_index=True)
rainfall_df.to_parquet(cfg.output_dir / "rainfall_raw.parquet")
print(f"Collected {len(rainfall_df):,} point-days. Citation: {provider.citation()}")


flood_events_path = Path(cfg.raw["data_sources"]["flood_events"]["path"])
if not flood_events_path.exists():
    raise FileNotFoundError(
        f"{flood_events_path} not found. This file must be created with "
        f"verified flood events (CWC/DFO/EM-DAT sourced only — see "
        f"documentation/flood_events_schema.md). The prior project's "
        f"33-station event list cannot be reused as-is: this study uses "
        f"basin-distributed points, not the same station identifiers, and "
        f"News-only events are excluded per config.yaml."
    )

flood_events_df = pd.read_csv(flood_events_path, parse_dates=["Start", "End"])
allowed_sources = set(cfg.raw["data_sources"]["flood_events"]["sources_allowed"])
bad_rows = flood_events_df[~flood_events_df["Source"].isin(allowed_sources)]
if len(bad_rows) > 0:
    raise ValueError(
        f"{len(bad_rows)} flood events use a Source not in {allowed_sources}. "
        f"Fix data/flood_events_basins.csv before proceeding."
    )
print(f"Loaded {len(flood_events_df)} verified flood events")


from floodai.features.pipeline import (
    add_temporal_features, add_rainfall_window_features,
    add_scs_cn_runoff, add_interaction_features,
)

df = rainfall_df.merge(points_df[["point_id", "lat", "lon", "basin_key"]], on=["point_id", "basin_key"])
df = df.sort_values(["point_id", "Date"]).reset_index(drop=True)
df = add_temporal_features(df)
df = add_rainfall_window_features(df, group_col="point_id")
# CN/elevation/humidity/temperature joins to df go here once Step 2's IMD
# fetch is confirmed working end-to-end and terrain covariates (gis/terrain.py)
# have been run against real SRTM tiles for these basins.
print(f"Feature matrix: {df.shape}")


# TODO (fill in after Steps 0-4 are verified against real IMD data):
#   1. label_floods(df, flood_events_df) -> df['Flood_Occurred']
#   2. temporal train/val/test split using cfg.split
#   3. RobustScaler fit on train only
#   4. floodai.training.imbalance.resample_training_only(...)
#   5. floodai.training.tuning.run_optuna_search(...)
#   6. floodai.training.threshold.select_f1_optimal_threshold(...)
#   7. floodai.evaluation.metrics.evaluate(..., provenance=DataProvenance.HELD_OUT)
#      then floodai.evaluation.metrics.report_headline(...)
#   8. floodai.training.lobo.run_lobo_cv(...)
raise NotImplementedError(
    "Steps 5+ intentionally not pre-filled against real data. "
    "See tests/test_integration.py for the exact call sequence to follow."
)


